# Module 2 — Text Summarization
**Pipeline:** IR → **Summarization** → QA  
**Datasets:** CNN/DailyMail (train + eval) · XSum (cross-dataset generalization eval)  
**Goal:** Compress retrieved passages into key sentences before QA  
**Method:** Hybrid (TF-IDF + TextRank + Position weighting)
**Evaluation:** ROUGE-1, ROUGE-2, ROUGE-L

## 1. Install & Import

In [1]:
!pip install datasets rouge-score networkx nltk -q

import re, os, json, math
import numpy as np
import networkx as nx
import nltk
from nltk.tokenize import sent_tokenize
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('All imports done.')

All imports done.


## 2. Load Datasets

In [2]:
# CNN/DailyMail — primary training & evaluation dataset
print('Loading CNN/DailyMail...')
cnn_dm = load_dataset('abisee/cnn_dailymail', '3.0.0')
train_data = cnn_dm['train']
val_data   = cnn_dm['validation']
test_data  = cnn_dm['test']

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print('Columns:', train_data.column_names)

Loading CNN/DailyMail...


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Train: 287113 | Val: 13368 | Test: 11490
Columns: ['article', 'highlights', 'id']


In [3]:
# XSum — cross-dataset generalization evaluation
print('Loading XSum...')
xsum      = load_dataset('EdinburghNLP/xsum')
xsum_test = xsum['test']
print(f'XSum test: {len(xsum_test)} examples')
print('Columns:', xsum_test.column_names)

Loading XSum...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

XSum test: 11334 examples
Columns: ['document', 'summary', 'id']


In [4]:
# Inspect samples
sample = train_data[0]
print('--- CNN/DailyMail sample ---')
print('Article (first 300 chars):', sample['article'][:300])
print('\nHighlights:', sample['highlights'][:300])

print('\n--- XSum sample ---')
xs = xsum_test[0]
print('Document (first 300 chars):', xs['document'][:300])
print('\nSummary:', xs['summary'])

--- CNN/DailyMail sample ---
Article (first 300 chars): LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi

Highlights: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .

--- XSum sample ---
Document (first 300 chars): Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than

Summary: There is a "chronic" need for more housing for priso

## 3. Text Preprocessing

In [5]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\(CNN\)', '', text)
    text = re.sub(r'--', ' ', text)
    return text

def tokenize_sentences(text):
    text = clean_text(text)
    return sent_tokenize(text)

def preprocess_sentence(sentence):
    tokens = re.findall(r'\b[a-z]+\b', sentence.lower())
    return [t for t in tokens if t not in STOPWORDS]

# Test
sents = tokenize_sentences(train_data[0]['article'])
print(f'Sentence count: {len(sents)}')
print('First sentence:', sents[0])

Sentence count: 24
First sentence: LONDON, England (Reuters)   Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him.


In [6]:
# Fit corpus-level TF-IDF vectorizer once across 5000 training articles
print('Fitting corpus-level TF-IDF vectorizer...')
sample_articles = [item['article'] for item in list(train_data)[:5000]]
sample_articles_clean = [' '.join(preprocess_sentence(a)) for a in sample_articles]
corpus_vectorizer = TfidfVectorizer(max_features=30_000)
corpus_vectorizer.fit(sample_articles_clean)
print('Corpus vectorizer ready.')

Fitting corpus-level TF-IDF vectorizer...
Corpus vectorizer ready.


## 6. Method C — Position-Weighted Hybrid

In [7]:
def summarize_hybrid(text, ratio=0.3, min_sents=2, max_sents=5,
                     alpha=0.4, beta=0.4, gamma=0.2):
    sentences = tokenize_sentences(text)
    n = len(sentences)
    if n <= min_sents:
        return ' '.join(sentences)

    clean = [' '.join(preprocess_sentence(s)) for s in sentences]

    # --- TF-IDF scores (corpus-level) ---
    try:
        mat = corpus_vectorizer.transform(clean).toarray()
    except ValueError:
        return sentences[0]
    tfidf_scores = mat.sum(axis=1) / (mat.astype(bool).sum(axis=1) + 1e-9)

    # --- TextRank scores ---
    sim_mat = cosine_similarity(mat)          # reuse mat — no extra function needed
    np.fill_diagonal(sim_mat, 0)
    graph = nx.from_numpy_array(sim_mat)
    try:
        tr_dict = nx.pagerank(graph, max_iter=200)
        tr_scores = np.array([tr_dict[i] for i in range(n)])
    except Exception:
        tr_scores = np.ones(n) / n

    # --- Position scores ---
    pos_scores = np.zeros(n)
    for i in range(n):
        if i < max(1, n // 5):
            pos_scores[i] = 1.0
        elif i >= n - max(1, n // 10):
            pos_scores[i] = 0.5

    # --- Normalize and combine ---
    def norm(arr):
        mx = arr.max()
        return arr / mx if mx > 0 else arr

    combined = alpha * norm(tfidf_scores) + beta * norm(tr_scores) + gamma * pos_scores
    n_keep   = max(min_sents, min(max_sents, int(n * ratio)))
    top_idx  = sorted(np.argsort(combined)[::-1][:n_keep])
    return ' '.join(sentences[i] for i in top_idx)

# Demo
article   = train_data[5]['article']
reference = train_data[5]['highlights']
summary_h = summarize_hybrid(article)
print('=== Hybrid Summary ===')
print(summary_h[:400])

=== Hybrid Summary ===
BAGHDAD, Iraq    Dressed in a Superman shirt, 5-year-old Youssif held his sister's hand Friday, seemingly unaware that millions of people across the world have been touched by his story. Youssif holds his sister's hand Friday. It's the best birthday present the Iraqi family could ever have imagined for their boy: Youssif turns 6 next Friday. No one has been arrested or held accountable in Youssif'


## 7. ROUGE Evaluation 

In [8]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeLsum'], use_stemmer=True)

def evaluate(summarize_fn, dataset, n_samples=1000,
             article_key='article', reference_key='highlights', label=''):
    r1, r2, rl = [], [], []
    for item in list(dataset)[:n_samples]:
        article   = item[article_key]
        reference = item[reference_key]
        if not article.strip() or not reference.strip():
            continue
        pred   = summarize_fn(article)
        scores = scorer.score(reference, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeLsum'].fmeasure)

    print(f'\n=== {label} (n={len(r1)}) ===')
    print(f'  ROUGE-1 : {np.mean(r1):.4f}')
    print(f'  ROUGE-2 : {np.mean(r2):.4f}')
    print(f'  ROUGE-L : {np.mean(rl):.4f}')
    return {'R1': round(np.mean(r1),4),
            'R2': round(np.mean(r2),4),
            'RL': round(np.mean(rl),4)}

# CNN/DailyMail — 1000 samples
cnn_results = evaluate(
    summarize_hybrid, val_data,
    n_samples=1000,
    label='Hybrid — CNN/DailyMail'
)

# XSum — 1000 samples (generalization)
xsum_results = evaluate(
    summarize_hybrid, xsum_test,
    n_samples=1000,
    article_key='document',
    reference_key='summary',
    label='Hybrid — XSum (generalization)'
)


=== Hybrid — CNN/DailyMail (n=1000) ===
  ROUGE-1 : 0.2757
  ROUGE-2 : 0.1045
  ROUGE-L : 0.2233

=== Hybrid — XSum (generalization) (n=1000) ===
  ROUGE-1 : 0.1817
  ROUGE-2 : 0.0295
  ROUGE-L : 0.1211


## 9. Pipeline Integration — Summarize IR Results

In [9]:
def summarize_passages(passages):
    return [summarize_hybrid(p) for p in passages]

os.makedirs('/kaggle/working/summ_artifacts', exist_ok=True)
with open('/kaggle/working/summ_artifacts/config.json', 'w') as f:
    json.dump({'method': 'hybrid', 'ratio': 0.3, 'max_sents': 5}, f)

print('Summarization artifacts saved.')

Summarization artifacts saved.


## 10. Live Demo

In [10]:
def summ_demo(article_text, reference=None):
    print('\n--- ORIGINAL (first 400 chars) ---')
    print(article_text[:400])
    print(f'  [{len(tokenize_sentences(article_text))} sentences]')

    summary = summarize_hybrid(article_text)
    print('\n--- HYBRID SUMMARY ---')
    print(summary)
    print(f'  [{len(tokenize_sentences(summary))} sentences]')

    if reference:
        s = scorer.score(reference, summary)
        print(f'\nROUGE-1: {s["rouge1"].fmeasure:.4f} | '
              f'ROUGE-2: {s["rouge2"].fmeasure:.4f} | '
              f'ROUGE-L: {s["rougeLsum"].fmeasure:.4f}')

sample = test_data[42]
summ_demo(sample['article'], reference=sample['highlights'])


--- ORIGINAL (first 400 chars) ---
Atlanta (CNN)Robert Lewis Burns Jr., the original drummer in Southern rock band Lynyrd Skynyrd, died Friday night in a car crash, a Georgia State Patrol spokesman said. Burns, 64, died after his car hit a mailbox and a tree in Cartersville, spokesman James Tallent said. No other cars were involved in the crash, which occurred shortly before midnight. "He was not restrained at the time of the crash
  [17 sentences]

--- HYBRID SUMMARY ---
Atlanta Robert Lewis Burns Jr., the original drummer in Southern rock band Lynyrd Skynyrd, died Friday night in a car crash, a Georgia State Patrol spokesman said. Burns, 64, died after his car hit a mailbox and a tree in Cartersville, spokesman James Tallent said. No other cars were involved in the crash, which occurred shortly before midnight. He was a great great drummer." People we've lost in 2015 .
  [5 sentences]

ROUGE-1: 0.3871 | ROUGE-2: 0.2637 | ROUGE-L: 0.3441
